# 2. Where the leakage actually is

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PhilanthroPy-Project/PhilanthroPy/blob/main/examples/notebooks/02_temporal_leakage.ipynb)

Two questions, measured rather than asserted:

**A. Does the cross-validation splitter matter?** Random `StratifiedKFold`
against walk-forward `FiscalYearGroupedSplitter`, both compared to a genuinely
held-out future year.

**B. Does *when* you build the features matter?** The same walk-forward split,
run once on aggregates computed **as of** each year and once on the same
aggregates computed over the **whole** export, including years that had not
happened yet.

The received wisdom says A is the important one. It is not.

In [ ]:
try:
    from philanthropy.datasets import make_donor_panel
except ImportError:
    # make_donor_panel is newer than the current PyPI release.
    !pip install -q "philanthropy @ git+https://github.com/PhilanthroPy-Project/PhilanthroPy@main"
    from philanthropy.datasets import make_donor_panel

import philanthropy
print("philanthropy", philanthropy.__version__)

## A panel, not a flat table

`make_donor_panel` returns what a fundraising shop actually exports: one row per
**gift**, with a fiscal year attached. Nothing is pre-aggregated, because
aggregation is the subject.

In [ ]:
import numpy as np
import pandas as pd

N_DONORS = 1500
YEARS = list(range(2018, 2025))

panel = make_donor_panel(
    n_donors=N_DONORS, n_years=len(YEARS), start_fiscal_year=YEARS[0], random_state=42
)
gifts = panel["gifts"]
gifts.head()

## Build the features twice

Identical aggregates. The only difference is the slice they are computed over.

`as_of` uses `amount[:, :j+1]`, everything up to and including the year being
scored. `whole` uses `amount`, the entire export. That single character is the
whole experiment, and it is the mistake this library exists to prevent: build
features once over the full history, split afterwards, and the future has
already leaked into every training row.

In [ ]:
amount = np.zeros((N_DONORS, len(YEARS)))
amount[gifts["donor_id"], gifts["fiscal_year"] - YEARS[0]] = gifts["gift_amount"]
gave = amount > 0

FEATURES = ["total", "n", "recent"]


def build(as_of):
    frames = []
    for j, year in enumerate(YEARS[:-1]):
        common = dict(
            fy=year,
            recent=amount[:, j],
            y=gave[:, j + 1].astype(int),   # gave in the FOLLOWING year
        )
        if as_of:
            frames.append(pd.DataFrame(dict(
                total=amount[:, : j + 1].sum(1), n=gave[:, : j + 1].sum(1), **common
            )))
        else:
            frames.append(pd.DataFrame(dict(
                total=amount.sum(1), n=gave.sum(1), **common
            )))
    return pd.concat(frames, ignore_index=True)


as_of_df, whole_df = build(True), build(False)
print(as_of_df.shape, whole_df.shape)
as_of_df.head(3)

## The honest target

"Train on everything before the final year, score the final year." That is the
number every backtest is trying to estimate, so measure it directly and compare
the estimators against it.

Both cross-validations below **exclude** the final year, for the same reason: it
is the estimand. Leaving it in would put it inside one estimator and not the
other, and walk-forward would win by construction rather than on merit.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

from philanthropy.model_selection import FiscalYearGroupedSplitter


def clf():
    return RandomForestClassifier(n_estimators=100, random_state=0)


def true_future(df):
    X, y, fy = df[FEATURES].to_numpy(), df["y"].to_numpy(), df["fy"].to_numpy()
    train, test = fy < fy.max(), fy == fy.max()
    fitted = clf().fit(X[train], y[train])
    return roc_auc_score(y[test], fitted.predict_proba(X[test])[:, 1])


def cv(df, splitter, use_groups):
    df = df[df["fy"] < df["fy"].max()]
    X, y, fy = df[FEATURES].to_numpy(), df["y"].to_numpy(), df["fy"].to_numpy()
    kwargs = {"groups": fy} if use_groups else {}
    return cross_val_score(
        clf(), X, y, cv=splitter, scoring="roc_auc", **kwargs
    ).mean()


truth = true_future(as_of_df)
walk = cv(as_of_df, FiscalYearGroupedSplitter(n_splits=3, drop_repeat_donors=False), True)
random = cv(as_of_df, StratifiedKFold(5, shuffle=True, random_state=0), False)
leaky = cv(whole_df, FiscalYearGroupedSplitter(n_splits=3, drop_repeat_donors=False), True)

## Results

In [ ]:
pd.DataFrame(
    {
        "ROC-AUC": [truth, walk, random],
        "error vs the true future": [np.nan, walk - truth, random - truth],
    },
    index=[
        "true future, final year held out",
        "walk-forward FiscalYearGroupedSplitter",
        "random StratifiedKFold",
    ],
).round(3)

In [ ]:
pd.DataFrame(
    {"ROC-AUC": [walk, leaky, leaky - walk]},
    index=[
        "features built as of each year",
        "same features over the whole export",
        "INFLATION",
    ],
).round(3)

## What that says

Same model, same splitter, same label. The only difference in the second table
is *when* the aggregate was computed, and it is worth several times what the
splitter choice is worth. **No choice of splitter recovers it.**

That is why every fitted statistic in this package is computed in `fit` and
frozen before `transform`, and why `EncounterTransformer` and
`GratefulPatientFeaturizer` take an `as_of` cutoff.

These are synthetic numbers on a generator whose persistence and drift were
chosen. On a **real** donor file, KDD Cup 1998, 95,412 donors, the same
experiment gives **+0.376 AUC** of inflation, roughly three times the synthetic
effect, against +0.107 for the splitter. Full write-up:
[Real-data replication](https://philanthropy-project.github.io/PhilanthroPy/explanation/real_data_replication/).

## Optional: reproduce the real-data number

The cell below downloads KDD Cup 1998 (~36 MB) from the UCI mirror and takes a
few minutes. It is off unless you ask for it, and it is the only function in
this package that touches the network.

In [ ]:
import os

if os.environ.get("PHILANTHROPY_FETCH_KDD98"):
    from philanthropy.datasets import fetch_kdd98_donors

    real = fetch_kdd98_donors()
    print(f"{len(real):,} donors, {real.shape[1]} columns")
    print("Full experiment: python scripts/real_data_leakage_experiment.py")
else:
    print("Skipped. Set PHILANTHROPY_FETCH_KDD98=1 to run it.")

## Next

**[3. A grateful-patient pipeline](03_grateful_patient_pipeline.ipynb)**, where
the `as_of` cutoff stops being an argument and starts being a parameter you
have to pass.